# Transport Operations MIS & Performance Analytics System
## Notebook 02: Data Validation & Quality Audit
**Organization**: Raftaar Cargo & Transport Services Pvt. Ltd.  
**Objective**: Implement automated validation checks to enforce business logic: non-negative freight, positive distance, referential integrity against vehicle & driver registries, and date chronology. Outputs the official **Data Quality Report**.

In [1]:
import pandas as pd
import json

df_shipments = pd.read_csv('../data/processed/cleaned_shipments.csv')
df_vehicles = pd.read_csv('../data/raw/vehicles.csv')
df_drivers = pd.read_csv('../data/raw/drivers.csv')
print("Validation libraries loaded.")

Validation libraries loaded.


### 1. Business Logic Rules Execution
- **Rule 1**: Shipment_ID not null or empty
- **Rule 2**: Freight > 0
- **Rule 3**: Distance_KM > 0
- **Rule 4**: Chronology (`Actual_Delivery >= Booking_Date`)
- **Rule 5**: Foreign keys exist in Vehicles & Drivers masters

In [2]:
valid_vehicles = set(df_vehicles['Vehicle_ID'].dropna().unique())
valid_drivers = set(df_drivers['Driver_ID'].dropna().unique())

rule_freight = df_shipments['Freight'] > 0
rule_distance = df_shipments['Distance_KM'] > 0
rule_chronology = (pd.to_datetime(df_shipments['Actual_Delivery']) >= pd.to_datetime(df_shipments['Booking_Date'])) | df_shipments['Actual_Delivery'].isna()
rule_vehicle_fk = df_shipments['Vehicle_ID'].isin(valid_vehicles)
rule_driver_fk = df_shipments['Driver_ID'].isin(valid_drivers)

valid_mask = rule_freight & rule_distance & rule_chronology & rule_vehicle_fk & rule_driver_fk
df_validated = df_shipments[valid_mask].copy()

print("--- Validation Results ---")
print(f"Total Cleaned Records: {len(df_shipments)}")
print(f"Negative/Zero Freight Failures: {(~rule_freight).sum()}")
print(f"Negative/Zero Distance Failures: {(~rule_distance).sum()}")
print(f"Chronology / Negative Transit Days: {(~rule_chronology).sum()}")
print(f"Foreign Key Mismatches: {(~(rule_vehicle_fk & rule_driver_fk)).sum()}")
print(f"Passed Records: {len(df_validated)} ({round(len(df_validated)/len(df_shipments)*100, 2)}% compliance)")

df_validated.to_csv('../data/processed/validated_shipments.csv', index=False)

--- Validation Results ---
Total Cleaned Records: 32
Negative/Zero Freight Failures: 2
Negative/Zero Distance Failures: 1
Chronology / Negative Transit Days: 1
Foreign Key Mismatches: 1
Passed Records: 30 (93.75% compliance)
